In [0]:
%run ./01-config


In [0]:
from pyspark.sql import functions as F

#base_dir_data defined in 01-config
landing_zone = base_dir_data + "/raw"
checkpoint_base = base_dir_checkpoint + "/checkpoints"

spark.sql(f"CREATE DATABASE IF NOT EXISTS {catalog}.{db_name}") 
spark.sql(f"USE {catalog}.{db_name}")   #will need to define catalog and db_name in function arg pulling all functions together (in 07-run script)


#data consumption functions for each table 
# registered_user_bz    --> consume_user_registration()
# gym_login_bz          --> consume_gym_logins()
# kafka_multiplex_bz    --> consume_kafka_multiplex()

#------------------------------------------------------
# CONSUME DATA INTO BRONZE LAYER 
#------------------------------------------------------

#registered_user_bz 
def consume_user_registration(once = True, processing_time = "5 seconds"): #run the stream once then stop, OR,  process every 5 seconds 
    #when calling func will have to specify once = False to run continuously

    #define schema - telling Auto Loader what columns the csv has (csv in sbit-unmanaged-dev/data-zone/raw/registered_users_bz)
    schema = "user_id long, device_id long, mac_address string, registration_timestamp double" #long = 64-bit Integer, string = text, double = float
    #long (BIGINT in SQL) is 64-bit Integer - standard type for Pyspark

    #monitor landing zone (raw storage area) & read new csv files into the stream
    #read stream using Auto Loader
    #creates a streaming DataFrame of incoming data
    df_stream = (spark.readStream 
                    .format("cloudfiles") # cloudfiles is a format that auto loader uses to read files from cloud storage (only format that activates auto loader)
                    #This is where it keeps track files that arrived, and processes new files as they arrive
                    #will only use cloudfiles in bronze layer
                    .schema(schema) # csv file headings defined above 
                    .option("cloudFiles.maxFilesPerTrigger", 1) # just one file per run to not overload
                    .option("cloudFiles.format", "csv")         
                    .option("header", True)                     # file has headers
                    .load(landing_zone + "/registered_users_bz") # monitoring raw data folder location (sbit-unmanaged-dev/data-zone/raw/registered_users_bz)
                    #if there are any new files the auto loader will pick it up and process it
                    .withColumn("load_time", F.current_timestamp())  # adding metadata - when the file was loaded
                    .withColumn("source_file", F.input_file_name())  # adding metadata - exact csv file that was loaded
                    # add new columns .withcolumn(name, data) 
                )



    #write to delta table Bronze layer of DB - registered_users_bz
    #write meta data to checkpoint 
    stream_writer = (df_stream.writeStream 
        .format("delta") # Delta table: directory of Parquet files that persist, like a SQL table 
        .option("checkpointLocation", checkpoint_base + "/registered_users_bz") # metadata storage - logs of all files that have already been moved to avoid duplicates
        .outputMode("append") #Use append mode because bronze layer is expected to insert only from source (always append in bronze layer)
        .queryName("dev.sbit_db.registered_users_bz")
    )

    #if you delete the checkpoint folder, the auto loader will think that nothing has been loaded and all files will be ingested again 
  

    #add if else statement 
    if once == True: #run only once (defined in func param)
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.registered_users_bz")
    else: 
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.registered_users_bz")  #continuous stream 
    

# gym_login_bz
def consume_gym_logins(once = True, processing_time = "5 seconds"):
    schema = "mac_address string, gym bigint, login double, logout double"

    df_stream = (spark.readStream 
                    .format("cloudFiles") 
                    .schema(schema) 
                    .option("maxFilesPerTrigger", 1) 
                    .option("cloudFiles.format", "csv") 
                    .option("header", "true") 
                    .load(landing_zone + "/gym_logins_bz") 
                    .withColumn("load_time", F.current_timestamp())
                    .withColumn("source_file", F.input_file_name())
                )

    stream_writer = (df_stream.writeStream 
                                .format("delta") 
                                .option("checkpointLocation", checkpoint_base + "/gym_logins_bz") 
                                .outputMode("append")     # Use append for bronze layer
                                .queryName("gym_logins_bz_ingestion_stream")
                    )       

    
    if once == True:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.gym_logins_bz")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.gym_logins_bz")


# kafka_multiplex_bz
def consume_kafka_multiplex(once = True, processing_time = "5 seconds"):
    schema = "key string, value string, topic string, partition bigint, offset bigint, timestamp bigint"
    #data coming from 3 json files (user_info, bpm, workout) within the kafka multiplex folder. 
    #They all have the same schema, so can use this schema variable for all 3.
   
    #timestamp is in milliseconds, so needs to be converted to a date format (to seconds - then to date)
    df_date_lookup = spark.table(f"{catalog}.{db_name}.date_lookup").select("date", "week_part")   # load date lookup table - "date" and "week_part" selected

    df_stream = (spark.readStream   # readstream = data stream reader, df_stream = df object (streaming df)
                    .format("cloudFiles")
                    .schema(schema)
                    .option("maxFilesPerTrigger", 5)
                    .option("cloudFiles.format", "json")
                    .load(landing_zone + "/kafka_multiplex_bz")       # once this is loaded it is a streaming df as readStream has data loaded                 
                    .withColumn("load_time", F.current_timestamp())       
                    .withColumn("source_file", F.input_file_name())
                    .join(F.broadcast(df_date_lookup), # joining date lookup table - ie. the "date" and "week_part" columns selected above are getting added 
                                                       # broadcast is an efficient one-to-many join during streaming (when data is relatively small (here 365 rows per year only))
                            F.to_date((F.col("timestamp")/1000).cast("timestamp")) == F.col("date"),  # join condition (if more than 1 condition, use list)
                            "left")  # left join (kafka record will always remain even if issue with date table)
                            #timestamp/1000 --> covert milliseconds to seconds
                            #.cast("timestamp") --> creates Spark timestamp (YYYY-MM-DD HH:MM:SS)
                            #F.to_date() --> Extracts just the date (YYYY-MM-DD)
                )
    
    #join syntax: 
    # df1.join(df2, join_condition, join_type)
    # df1.join(df2, [join_condition1, join_condition2], join_type)
    # df_stream.join(
                    # F.broadcast(df_date_lookup),                                              #df to join
                    # F.to_date((F.col("timestamp")/1000).cast("timestamp")) == F.col("date"),  #join condition
                    # "left"                                                                    #join type
                    # ) 
    
    #you do not usually amend tables in bronze layer but it is done here because: 
    # - just adding date lookup (common)
    # - it is metadata enrichment only 
    # - no change to raw values / busines logic
    # - it will help downstream silver tables

    stream_writer = (df_stream.writeStream 
                                .format("delta") 
                                .option("checkpointLocation", checkpoint_base + "/kafka_multiplex_bz") 
                                .outputMode("append")  # Use append for bronze layer
                                .queryName("kafka_multiplex_bz_ingestion_stream")
                    )
    

    if once == True:
        return stream_writer.trigger(availableNow=True).toTable(f"{catalog}.{db_name}.kafka_multiplex_bz")
    else:
        return stream_writer.trigger(processingTime=processing_time).toTable(f"{catalog}.{db_name}.kafka_multiplex_bz")
    

#--------------------------------------------------------------------
# Function to call all functions (in 07-run script)
#--------------------------------------------------------------------  

def consume_bronze(once=True, processing_time="5 seconds"):
    """
    - start time and print statement
    - call all functions
    - if once == True, wait for all streams to finish
    - print statement """
    import time
    start = int(time.time())
    print(f"\nStarting bronze layer consumption ...")
    consume_user_registration(once, processing_time) 
    consume_gym_logins(once, processing_time) 
    consume_kafka_multiplex(once, processing_time)
    if once:      # same as if once == True - like above 
        for stream in spark.streams.active:     # spark.streams.activate : list of all currently running streaming jobs
            stream.awaitTermination()  #finish the one before before moving to the next table (should queue). Also does not exit until all streams are completed. 
        #if once == False  the above block would be skipped and it would be continuously streaming anyway 
    print(f"Completed bronze layer consumtion {int(time.time()) - start} seconds")

#--------------------------------------------------------------------
# Validation
#-------------------------------------------------------------------- 

def assert_count(catalog, db_name, table_name, expected_count, filter="true"): # filter true - for kafka part of the stream : "topic='user_info'" when calling 
    print(f"Validating record counts in {table_name}...", end='')
    actual_count = spark.read.table(f"{catalog}.{db_name}.{table_name}").where(filter).count() #filters here and then counts 
    assert actual_count == expected_count, f"Expected {expected_count:,} records, found {actual_count:,} in {table_name} where {filter}" 
    print(f"Found {actual_count:,} / Expected {expected_count:,} records where {filter}: Success")        
    


def validate_bronze(catalog, db_name, sets): # sets - eg. registered_users_1 or 2 
    import time
    start = int(time.time())
    print(f"\nValidating bronz layer records...")
    #there are 5 records(rows) in the registered_users_bz file (check base file)
    assert_count(catalog, db_name,"registered_users_bz", 5 if sets == 1 else 10)# if set 1 then 5 records in file, else if set 2 as well we expect 10 records
    assert_count(catalog, db_name,"gym_logins_bz", 8 if sets == 1 else 16) #filter is true by default for these first 2 tables (see assert_count() above)
    #3 tables come from the kafka_multiplex so add in "topic" filter to specify which table you are checking
    assert_count(catalog, db_name,"kafka_multiplex_bz", 7 if sets == 1 else 13, "topic='user_info'")
    assert_count(catalog, db_name,"kafka_multiplex_bz", 16 if sets == 1 else 32, "topic='workout'")
    assert_count(catalog, db_name,"kafka_multiplex_bz", sets * 253801, "topic='bpm'") # bpm is a huge file with 253801 records (eg. 1 * 253801, 2 * 253801)
    print(f"Bronze layer validation completed in {int(time.time()) - start} seconds") 

    #ok to hard code number of records in dev environment, in production would need to adapt 